# Load library and path

In [1]:
import os
import numpy as np
import subprocess

# Base directory (portable)
base_dir = os.getcwd()
print("Working directory:", base_dir)

# Folders
dna_dir = os.path.join(base_dir, "dna")
pdbqt_dir = os.path.join(base_dir, "pdbqt")
output_dir = os.path.join(base_dir, "output")
template_dir = os.path.join(base_dir, "template")

# Files
vina_exe = os.path.join(base_dir, "vina.exe")
ligand_pdb = os.path.join(template_dir, "template.pdb")
ligand_pdbqt = os.path.join(template_dir, "template.pdbqt")

Working directory: C:\Users\cnguyen\Desktop\MD_DNA_PQS


# Check environment

In [2]:
print("DNA files:", os.listdir(dna_dir))
print("PDBQT folder:", os.listdir(pdbqt_dir))
print("Ligand PDB exists:", os.path.exists(ligand_pdb))
print("Vina exists:", os.path.exists(vina_exe))

DNA files: ['seq1.pdb', 'seq10.pdb', 'seq11.pdb', 'seq12.pdb', 'seq13.pdb', 'seq14.pdb', 'seq15.pdb', 'seq2.pdb', 'seq3.pdb', 'seq4.pdb', 'seq5.pdb', 'seq6.pdb', 'seq7.pdb', 'seq8.pdb', 'seq9.pdb']
PDBQT folder: ['seq1.pdbqt', 'seq10.pdbqt', 'seq11.pdbqt', 'seq12.pdbqt', 'seq13.pdbqt', 'seq14.pdbqt', 'seq15.pdbqt', 'seq2.pdbqt', 'seq3.pdbqt', 'seq4.pdbqt', 'seq5.pdbqt', 'seq6.pdbqt', 'seq7.pdbqt', 'seq8.pdbqt', 'seq9.pdbqt']
Ligand PDB exists: True
Vina exists: True


# Convert ligand / template into PDBQT

In [3]:
if not os.path.exists(ligand_pdbqt):
    print("🔄 Converting ligand to PDBQT...")
    os.system(f'obabel "{ligand_pdb}" -O "{ligand_pdbqt}" --partialcharge gasteiger')
    print(f"✅ Done")
else:
    print("✅ Ligand already converted")

✅ Ligand already converted


# Convert all DNA into PDBQT

In [4]:
for file in os.listdir(dna_dir):
    if file.endswith(".pdb") or file.endswith(".PDB"):
        
        name = os.path.splitext(file)[0]
        input_path = os.path.join(dna_dir, file)
        output_path = os.path.join(pdbqt_dir, name + ".pdbqt")
        
        if not os.path.exists(output_path):
            print(f"🔄 Converting {name} ✅ Done!")
            cmd = f'obabel "{input_path}" -O "{output_path}" --partialcharge gasteiger'
            os.system(cmd)
        else:
            print(f"✅ Already exists: {name}")

✅ Already exists: seq1
✅ Already exists: seq10
✅ Already exists: seq11
✅ Already exists: seq12
✅ Already exists: seq13
✅ Already exists: seq14
✅ Already exists: seq15
✅ Already exists: seq2
✅ Already exists: seq3
✅ Already exists: seq4
✅ Already exists: seq5
✅ Already exists: seq6
✅ Already exists: seq7
✅ Already exists: seq8
✅ Already exists: seq9


# Compute center function

In [5]:
def get_center(pdb_file):
    coords = []
    with open(pdb_file, "r") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords.append([x, y, z])
    coords = np.array(coords)
    return coords.mean(axis=0)

print(f"✅ Finished function to get center point!")

✅ Finished function to get center point!


# Fix all the DNA PBDQT files

In [6]:
def fix_receptor_pdbqt(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()
    
    cleaned = []
    for line in lines:
        if line.startswith(("ROOT", "ENDROOT", "BRANCH", "ENDBRANCH", "TORSDOF")):
            continue
        cleaned.append(line)
    
    with open(file_path, "w") as f:
        f.writelines(cleaned)

In [7]:
for file in os.listdir(pdbqt_dir):
    if file.endswith(".pdbqt"):
        path = os.path.join(pdbqt_dir, file)
        fix_receptor_pdbqt(path)
        print("Fixed:", file)

Fixed: seq1.pdbqt
Fixed: seq10.pdbqt
Fixed: seq11.pdbqt
Fixed: seq12.pdbqt
Fixed: seq13.pdbqt
Fixed: seq14.pdbqt
Fixed: seq15.pdbqt
Fixed: seq2.pdbqt
Fixed: seq3.pdbqt
Fixed: seq4.pdbqt
Fixed: seq5.pdbqt
Fixed: seq6.pdbqt
Fixed: seq7.pdbqt
Fixed: seq8.pdbqt
Fixed: seq9.pdbqt


## Test seq1

# Run all other sequences

In [8]:
results = []
size = [40, 40, 40]

for file in os.listdir(dna_dir):
    if file.endswith(".pdb"):
        
        name = file.replace(".pdb", "")
        
        pdb_path = os.path.join(dna_dir, file)
        pdbqt_path = os.path.join(pdbqt_dir, name + ".pdbqt")
        out_path = os.path.join(output_dir, name + "_out.pdbqt")
        
        if not os.path.exists(pdbqt_path):
            print("Missing:", name)
            continue
        
        center = get_center(pdb_path)
        cx, cy, cz = center
        
        print("Running:", name)  # short progress only
        
        cmd = [
            vina_exe,
            "--receptor", pdbqt_path,
            "--ligand", ligand_pdbqt,
            "--center_x", str(cx),
            "--center_y", str(cy),
            "--center_z", str(cz),
            "--size_x", str(size[0]),
            "--size_y", str(size[1]),
            "--size_z", str(size[2]),
            "--exhaustiveness", "8",
            "--num_modes", "10",
            "--out", out_path
        ]
        
        # 👇 suppress all output
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

Running: seq1
Running: seq10
Running: seq11
Running: seq12
Running: seq13
Running: seq14
Running: seq15
Running: seq2
Running: seq3
Running: seq4
Running: seq5
Running: seq6
Running: seq7
Running: seq8
Running: seq9


In [9]:
results = []

for file in os.listdir(output_dir):
    if file.endswith("_out.pdbqt"):
        
        name = file.replace("_out.pdbqt", "")
        path = os.path.join(output_dir, file)
        
        with open(path, "r") as f:
            for line in f:
                if "REMARK VINA RESULT" in line:
                    score = float(line.split()[3])
                    results.append((name, score))
                    break

results.sort(key=lambda x: x[1])

print("\n=== FINAL RANKING ===")
for r in results:
    print(r)


=== FINAL RANKING ===
('seq10', -6.809)
('seq8', -6.76)
('seq5', -6.717)
('seq9', -6.698)
('seq12', -6.666)
('seq13', -6.578)
('seq1', -6.576)
('seq4', -6.572)
('seq3', -6.553)
('seq7', -6.521)
('seq15', -6.519)
('seq11', -6.434)
('seq6', -6.408)
('seq14', -6.291)
('seq2', -5.996)
